In [ ]:
import json
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sentence_transformers import SentenceTransformer


# 1. Load your JSON
with open('data/waterloo-open-api-data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. Build a DataFrame of code, title, description
rows = []
for code, info in data.items():
    title = info.get('title', '')
    description = info.get('description', '')
    
    # If description is empty, use the title instead
    if not description or description.strip() == '':
        description = title
    
    rows.append({
        'courseCode': code,
        'title': title,
        'description': description
    })

df = pd.DataFrame(rows)
filtered = df[df['courseCode'].str.startswith(('ECE', 'MSE', 'MSCI'))]


# 3. TF‑IDF → SVD pipeline
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_tfidf = tfidf.fit_transform(df['description'])

svd = TruncatedSVD(n_components=100, random_state=42)

embeddings = svd.fit_transform(X_tfidf)

# 4. Attach & save
df['embedding'] = embeddings.tolist()
df.to_pickle('data/course_embeddings.pkl')
np.save('data/course_embeddings.npy', embeddings)

# 5. (Optional) Quick peek at the table
df[['courseCode','title','description']].head()


# 6. BERT embeddings
embedding_path = 'data/course_bert_embeddings.npy'

if not os.path.exists(embedding_path):

    # Load a BERT-based model (MiniLM is fast + accurate)
    model = SentenceTransformer('all-MiniLM-L6-v2')

    # Generate dense BERT embeddings (384-dim)
    descriptions = df['description'].fillna('').tolist()
    bert_embeddings = model.encode(descriptions, show_progress_bar=True)

    # Save embeddings
    np.save(embedding_path, bert_embeddings)
else:
    print(f"Embeddings file '{embedding_path}' already exists. Skipping embedding generation.")



KeyError: 'coursecodes'

In [ ]:
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import pandas as pd

# Reload your course DataFrame
df = pd.read_pickle('data/course_embeddings.pkl')  # assumes you re-run the vectorization cell

# 1. Fit TF‑IDF & SVD on your course descriptions
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_tfidf = tfidf.fit_transform(df['description'])

svd = TruncatedSVD(n_components=100, random_state=42)
embeddings = svd.fit_transform(X_tfidf)

# 2. Overwrite embeddings & save models
import numpy as np
df['embedding'] = embeddings.tolist()
df.to_pickle('data/course_embeddings.pkl')
np.save('data/course_embeddings.npy', embeddings)

with open('data/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open('data/svd_model.pkl', 'wb') as f:
    pickle.dump(svd, f)

print("Models and embeddings saved.")


Models and embeddings saved.


## Method 1: Direct Cosine‑Similarity Ranking

In [ ]:

import pickle, numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Load artifacts
df     = pd.read_pickle('data/course_embeddings.pkl')
emb    = np.load('data/course_embeddings.npy')
with open('data/tfidf_vectorizer.pkl','rb') as f: tfidf = pickle.load(f)
with open('data/svd_model.pkl','rb')         as f: svd   = pickle.load(f)

def recommend_cosine(query, df, top_k):
    q_vec = svd.transform(tfidf.transform([query]))               # (1,100)
    sims  = cosine_similarity(emb, q_vec).flatten()              # (N,)
    idxs  = sims.argsort()[::-1][:top_k]
    return df.iloc[idxs][['courseCode','title','description']].assign(similarity=sims[idxs])




## Method 2: Approximate Nearest‑Neighbor Search with FAISS

In [ ]:

import pickle, faiss, numpy as np, pandas as pd


# Load & normalize
df  = pd.read_pickle('data/course_embeddings.pkl')
emb = np.load('data/course_embeddings.npy').astype('float32')
faiss.normalize_L2(emb)

# Build or load index
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

with open('data/tfidf_vectorizer.pkl','rb') as f: tfidf = pickle.load(f)
with open('data/svd_model.pkl','rb')         as f: svd   = pickle.load(f)

def recommend_faiss(query, df, top_k):
    q = svd.transform(tfidf.transform([query])).astype('float32')
    faiss.normalize_L2(q)
    D, I = index.search(q, top_k)   # D=sim scores, I=indices
    return df.iloc[I[0]][['courseCode','title','description']].assign(similarity=D[0])




## Method 3: Diversity‑Aware Re‑Ranking (MMR)

In [ ]:

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# (Assumes tfidf, svd, emb, df already in scope)

def recommend_mmr(query, df, top_k, λ=0.7):
    q_vec      = svd.transform(tfidf.transform([query])).flatten()
    sims       = cosine_similarity(emb, q_vec.reshape(1,-1)).flatten()
    candidates = list(range(len(sims)))
    selected   = []

    for _ in range(top_k):
        if not selected:
            idx = int(np.argmax(sims))
        else:
            mmr_scores = []
            for i in candidates:
                rel = sims[i]
                red = max(cosine_similarity(emb[selected], emb[i].reshape(1,-1)).flatten())
                mmr_scores.append((i, λ*rel - (1-λ)*red))
            idx = max(mmr_scores, key=lambda x: x[1])[0]
        selected.append(idx)
        candidates.remove(idx)

    return df.iloc[selected][['courseCode','title','description']].assign(similarity=sims[selected])



## Method 4: Graph‑Based Diffusion (Personalized PageRank)

In [ ]:

import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# (Assumes tfidf, svd, emb, df already in scope)

def recommend_graph(query, df, top_k):
    q_vec = svd.transform(tfidf.transform([query]))               # (1,100)
    sims  = cosine_similarity(emb, q_vec).flatten()              # (N,)

    G = nx.DiGraph()
    G.add_node('query')
    for i, code in enumerate(df['courseCode']):
        G.add_edge('query', code, weight=float(sims[i]))

    pr = nx.pagerank(G, alpha=0.85, personalization={'query':1.0})
    ranked = sorted(
        ((c,sc) for c,sc in pr.items() if c!='query'),
        key=lambda x: x[1], reverse=True
    )[:top_k]
    codes  = [c for c,_ in ranked]
    scores = [s for _,s in ranked]
    return df[df['courseCode'].isin(codes)][['courseCode','title','description']].assign(score=scores)



## Method 5: Fuzzy String Matching with Multiple Fields

In [ ]:

from difflib import SequenceMatcher

def fuzzy_similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def recommend_fuzzy_multi(query, df, top_k, weights={'title': 0.4, 'description': 0.5, 'code': 0.1}):
    query_lower = query.lower()
    scores = []
    
    for idx, row in df.iterrows():
        title_sim = fuzzy_similarity(query_lower, str(row['title']))
        desc_sim = fuzzy_similarity(query_lower, str(row['description']))
        code_sim = fuzzy_similarity(query_lower, str(row['courseCode']))
        
        # Weighted combination
        total_score = (weights['title'] * title_sim + 
                      weights['description'] * desc_sim + 
                      weights['code'] * code_sim)
        
        scores.append((idx, total_score))
    
    # Sort by score
    scores.sort(key=lambda x: x[1], reverse=True)
    top_indices = [idx for idx, _ in scores[:top_k]]
    top_scores = [score for _, score in scores[:top_k]]
    
    return df.iloc[top_indices][['courseCode','title','description']].assign(fuzzy_score=top_scores)


# Method 6: Keyword Overlap with Position Weighting

In [ ]:

def extract_keywords(text, min_length=3):
    """Extract keywords, removing stop words and short terms"""
    import re
    stop_words = {'the', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'this', 'that', 'these', 'those', 'a', 'an'}
    words = re.findall(r'\b\w+\b', text.lower())
    return [w for w in words if len(w) >= min_length and w not in stop_words]

def recommend_keyword_overlap(query, df, top_k):
    query_keywords = set(extract_keywords(query))
    scores = []
    
    for idx, row in df.iterrows():
        title_keywords = set(extract_keywords(str(row['title'])))
        desc_keywords = set(extract_keywords(str(row['description'])))
        
        # Calculate overlaps
        title_overlap = len(query_keywords.intersection(title_keywords))
        desc_overlap = len(query_keywords.intersection(desc_keywords))
        
        # Position-weighted score (title matches worth more)
        total_keywords = len(title_keywords.union(desc_keywords))
        if total_keywords > 0:
            score = (2 * title_overlap + desc_overlap) / np.sqrt(total_keywords)
        else:
            score = 0
            
        scores.append((idx, score))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    top_indices = [idx for idx, _ in scores[:top_k]]
    top_scores = [score for _, score in scores[:top_k]]
    
    return df.iloc[top_indices][['courseCode','title','description']].assign(keyword_score=top_scores)

## Method 7: BERT embeddings

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Load BERT embeddings
bert_embeddings = np.load('data/course_bert_embeddings.npy')

def recommend_bert(query, df, top_k):
    query_embed = model.encode([query])
    sims = cosine_similarity(query_embed, bert_embeddings).flatten()
    top_indices = sims.argsort()[::-1][:top_k]
    return df.iloc[top_indices][['courseCode', 'title', 'description']].assign(bert_score=sims[top_indices])


## Method 8: Hybrid Ensemble Approach

In [ ]:


def recommend_hybrid_ensemble(query, df, top_k, method_weights=None):
    """Combine multiple methods with weighted voting"""
    if method_weights is None:
        method_weights = {
            'cosine': 0.2,
            'fuzzy': 0.2,
            'keyword': 0.2,
            'faiss': 0.2,
            'bert': 0.2
        }
    
    # Get results from different methods
    cosine_results = recommend_cosine(query, top_k=len(df))  # Get all for scoring
    fuzzy_results = recommend_fuzzy_multi(query, top_k=len(df))
    keyword_results = recommend_keyword_overlap(query, top_k=len(df))
    faiss_results = recommend_faiss(query, top_k=len(df))
    bert_results = recommend_bert(query, top_k=len(df))
    
    # Normalize scores for each method
    def normalize_scores(scores):
        scores = np.array(scores)
        if scores.max() != scores.min():
            return (scores - scores.min()) / (scores.max() - scores.min())
        return scores
    
    # Create combined scores
    combined_scores = {}
    
    for idx, code in enumerate(df['courseCode']):
        score = 0
        
        # Find scores from each method
        cosine_score = cosine_results[cosine_results['courseCode'] == code]['similarity'].iloc[0] if code in cosine_results['courseCode'].values else 0
        fuzzy_score = fuzzy_results[fuzzy_results['courseCode'] == code]['fuzzy_score'].iloc[0] if code in fuzzy_results['courseCode'].values else 0
        keyword_score = keyword_results[keyword_results['courseCode'] == code]['keyword_score'].iloc[0] if code in keyword_results['courseCode'].values else 0
        faiss_score = faiss_results[faiss_results['courseCode'] == code]['similarity'].iloc[0] if code in faiss_results['courseCode'].values else 0
        bert_score = bert_results[bert_results['courseCode'] == code]['bert_score'].iloc[0] if code in bert_results['courseCode'].values else 0
        
        # Weighted combination
        combined_score = (method_weights['cosine'] * cosine_score +
                         method_weights['fuzzy'] * fuzzy_score +
                         method_weights['keyword'] * keyword_score +
                         method_weights['faiss'] * faiss_score +
                         method_weights['bert'] * bert_score)
        
                        
        
        combined_scores[idx] = combined_score
    
    # Sort and return top results
    sorted_indices = sorted(combined_scores.keys(), key=lambda x: combined_scores[x], reverse=True)
    top_indices = sorted_indices[:top_k]
    top_scores = [combined_scores[idx] for idx in top_indices]
    
    return df.iloc[top_indices][['courseCode','title','description']].assign(hybrid_score=top_scores)

## Adding in Course dependancy

In [ ]:
import json
import pandas as pd
import ast

def extract_course_dependencies(json_file_path):
    """
    Extract course dependencies from JSON file into a DataFrame
    """
    # Load the JSON data
    with open(json_file_path, 'r', encoding='utf-8') as f:
        dependencies = json.load(f)
    
    # Initialize lists to store the extracted data
    course_codes = []
    prerequisite_courses = []
    prerequisite_groups = []
    program_requirements = []
    program_restrictions = []
    root_operators = []
    
    for course_code, dep_data in dependencies.items():
        course_codes.append(course_code)
        
        # Extract prerequisite courses
        prereq_courses = []
        prereq_groups = []
        
        for group in dep_data.get('groups', []):
            if group.get('type') == 'course':
                # Single course
                course_info = {
                    'code': group.get('code'),
                    'grade_requirement': group.get('grade_requirement'),
                    'name': group.get('name')
                }
                prereq_courses.append(course_info)
            elif group.get('type') == 'prerequisite_group':
                # Group of courses
                group_courses = []
                for course in group.get('courses', []):
                    course_info = {
                        'code': course.get('code'),
                        'grade_requirement': course.get('grade_requirement'),
                        'name': course.get('name')
                    }
                    group_courses.append(course_info)
                
                prereq_groups.append({
                    'courses': group_courses,
                    'operator': group.get('operator'),
                    'quantity': group.get('quantity')
                })
        
        prerequisite_courses.append(prereq_courses)
        prerequisite_groups.append(prereq_groups)
        program_requirements.append(dep_data.get('program_requirements', []))
        program_restrictions.append(dep_data.get('program_restrictions', []))
        root_operators.append(dep_data.get('root_operator', 'AND'))
    
    # Create DataFrame
    df = pd.DataFrame({
        'course_code': course_codes,
        'prerequisite_courses': prerequisite_courses,
        'prerequisite_groups': prerequisite_groups,
        'program_requirements': program_requirements,
        'program_restrictions': program_restrictions,
        'root_operator': root_operators
    })
    
    return df

def create_flattened_dependencies_df(df):
    """
    Create a flattened version where each prerequisite is on its own row
    """
    flattened_data = []
    
    for _, row in df.iterrows():
        course_code = row['course_code']
        
        # Add individual prerequisite courses
        for prereq in row['prerequisite_courses']:
            flattened_data.append({
                'course_code': course_code,
                'prerequisite_code': prereq['code'],
                'prerequisite_name': prereq['name'],
                'grade_requirement': prereq['grade_requirement'],
                'group_operator': None,
                'group_quantity': None,
                'prerequisite_type': 'individual_course',
                'program_requirements': row['program_requirements'],
                'program_restrictions': row['program_restrictions'],
                'root_operator': row['root_operator']
            })
        
        # Add prerequisite groups
        for group in row['prerequisite_groups']:
            for prereq in group['courses']:
                flattened_data.append({
                    'course_code': course_code,
                    'prerequisite_code': prereq['code'],
                    'prerequisite_name': prereq['name'],
                    'grade_requirement': prereq['grade_requirement'],
                    'group_operator': group['operator'],
                    'group_quantity': group.get('quantity'),
                    'prerequisite_type': 'group_course',
                    'program_requirements': row['program_requirements'],
                    'program_restrictions': row['program_restrictions'],
                    'root_operator': row['root_operator']
                })
    
    return pd.DataFrame(flattened_data)


    

print("Loading course dependencies...")
df_dependencies = extract_course_dependencies('data/course_dependencies/course_dependencies.json')

print(f"Extracted {len(df_dependencies)} courses with dependencies")
print("\nSample of the structured data:")
df_dependencies.head()


# Create flattened version
print("\nCreating flattened version...")
flattened_df = create_flattened_dependencies_df(df_dependencies)

flattened_df['prerequisite_code'] = flattened_df['prerequisite_code'].str.replace(' ', '')
flattened_df.head(10)



Loading course dependencies...
Extracted 280 courses with dependencies

Sample of the structured data:

Creating flattened version...


,course_code,prerequisite_code,prerequisite_name,grade_requirement,group_operator,group_quantity,prerequisite_type,program_requirements,program_restrictions,root_operator
0,MSE620,MSCI605,None,None,None,NaN,individual_course,[],[],AND
1,MSE630,MSCI605,None,None,None,NaN,individual_course,[],[],AND
2,MSE702,MSCI603,None,None,None,NaN,individual_course,[],[],AND
3,MSE703,MSCI603,None,None,OR,NaN,group_course,[],[],AND
4,MSE703,MSCI634,None,None,OR,NaN,group_course,[],[],AND
5,MSE719,MSCI603,None,None,OR,NaN,group_course,[],[],AND
6,MSE719,MSCI634,None,None,OR,NaN,group_course,[],[],AND
7,MSE741,MSCI607,None,None,None,NaN,individual_course,[],[],AND
8,MSE431,BME411,None,None,OR,1.0,group_course,[],"[{'type': 'program_restriction', 'program_name': 'Math', 'program_type': None, 'faculty': None, 'restriction_type': 'not_open'}]",AND
9,MSE431,CHE521,None,None,OR,1.0,group_course,[],"[{'type': 'program_restriction', 'program_name': 'Math', 'program_type': None, 'faculty': None, 'restriction_type': 'not_open'}]",AND


In [ ]:
import json
import pandas as pd
import ast




def check_prerequisites_flattened(course_code, flattened_df, completed_courses, grades=None):
    """
    Simplified version for your flattened DataFrame structure
    """
    # Get all prerequisite rows for this course
    course_prereqs = flattened_df[flattened_df['course_code'] == course_code]
    
    if course_prereqs.empty:
        return True, "No prerequisite information available"
    
    # Check if there are any prerequisites
    has_prereqs = course_prereqs[course_prereqs['prerequisite_code'].notna()]
    if has_prereqs.empty:
        return True, "No prerequisites required"
    
    # Handle individual courses (group_operator is None)
    individual_courses = has_prereqs[has_prereqs['group_operator'].isna()]
    for _, row in individual_courses.iterrows():
        prereq_code = row['prerequisite_code']
        if prereq_code not in completed_courses:
            return False, f"Missing prerequisite: {prereq_code}"
    
    # Handle group courses (group_operator is not None)
    group_courses = has_prereqs[has_prereqs['group_operator'].notna()]
    if not group_courses.empty:
        # Group by operator (assuming same operator = same group)
        for operator in group_courses['group_operator'].unique():
            group = group_courses[group_courses['group_operator'] == operator]
            
            if operator == 'OR':
                # Need at least one course from the group
                completed_in_group = 0
                for _, row in group.iterrows():
                    prereq_code = row['prerequisite_code']
                    if prereq_code in completed_courses:
                        completed_in_group += 1
                
                if completed_in_group == 0:
                    course_codes = group['prerequisite_code'].tolist()
                    return False, f"Need at least one from: {', '.join(course_codes)}"
            
            elif operator == 'AND':
                # Need ALL courses from the group
                for _, row in group.iterrows():
                    prereq_code = row['prerequisite_code']
                    if prereq_code not in completed_courses:
                        return False, f"Missing prerequisite: {prereq_code}"
    
    return True, "All prerequisites met"



def filter_eligible_courses(course_df, dependencies, completed_courses, grades=None):
    """
    Filter courses to show only those the student is eligible to take
    
    Args:
        course_df: DataFrame of all courses
        dependencies: Dependencies dictionary
        completed_courses: List of completed course codes
        grades: Dictionary of {course_code: grade}
    
    Returns:
        DataFrame of eligible courses with eligibility reasons
    """
    eligible_courses = []
    
    for _, course in course_df.iterrows():
        course_code = course['courseCode']
        
        # Skip if already completed
        if course_code in completed_courses:
            continue
        
        # Check prerequisites
        prereq_eligible, prereq_reason = check_prerequisites_flattened(
            course_code, dependencies, completed_courses, grades
        )
        
        
        if prereq_eligible:
            eligible_courses.append({
                'courseCode': course_code,
                'title': course['title'],
                'description': course['description'],
                'prerequisite_status': prereq_reason,
            })
    
    return pd.DataFrame(eligible_courses)


    





## Testing

### extract input from resume/transcript/input

In [ ]:

# Load the parsed resumes data
with open('data/parsed_resumes.json', 'r') as f:
    resumes = json.load(f)

# Extract subfields and preferred_domains for each resume
resume_fields = []

for i, resume in enumerate(resumes):
    # Get subfields from core_interests
    subfields = resume.get('core_interests', {}).get('subfields', [])
    
    # Get preferred_domains from learning_patterns
    preferred_domains = resume.get('learning_patterns', {}).get('preferred_domains', [])
    
    # Combine all fields for this resume
    all_fields_for_resume = subfields + preferred_domains
    
    resume_fields.append({
        'resume_index': i,
        'file_name': resume.get('file_name', ''),
        'fields': all_fields_for_resume
    })
resume_fields


[{'resume_index': 0,
  'file_name': 'Matthew_Ho_Resume_3A.pdf',
  'fields': ['Data Visualization',
   'Data Engineering',
   'Business Intelligence',
   'Business Operations',
   'Financial Planning',
   'Geospatial Analysis']},
 {'resume_index': 1,
  'file_name': 'Resume 6 E.pdf',
  'fields': ['Full-Stack Development',
   'Backend Systems',
   'Application Architecture',
   'Web Applications',
   'Data Management',
   'Cloud Services']},
 {'resume_index': 2,
  'file_name': 'Finance Sample Resume.pdf',
  'fields': ['Corporate Finance',
   'Financial Analysis',
   'Business Operations',
   'Financial Services']}]

In [ ]:
# Load data
course_df = df
dependencies = flattened_df

# Sample student data
completed_courses = ['CS146', 'MSE211', 'MSCI603', 'PHYS115']
grades = {
    'CS146': 85,
    'MSE211': 78,
    'MSCI603': 90,
    'PHYS115': 90
}

# Filter courses
eligible_df = filter_eligible_courses(course_df, dependencies, completed_courses, grades)

num = 1

search_queries = resume_fields[num]['fields']

methods = [
    ("cosine", recommend_cosine),
    ("faiss", recommend_faiss),
    # ("mmr", recommend_mmr),
    # ("graph", recommend_graph),
    ("fuzzy_multi", recommend_fuzzy_multi),
    # ("keyword_overlap", recommend_keyword_overlap),
    ("bert", recommend_bert)
]

# Prepare empty lists for dataframes to be filled
df_list = []

for i, search_query in enumerate(search_queries):
    for method_name, method_func in methods:
        try:
            results = method_func(query=search_query, df=eligible_df, top_k=3)
            # results is a DataFrame with courseCode, title, description columns
            for rank, (_, row) in enumerate(results.iterrows(), 1):
                # Check if score column exists in index (columns)
                score_col = None
                for col in ['similarity', 'fuzzy_score', 'keyword_score', 'bert_score', 'hybrid_score', 'score']:
                    if col in row.index:
                        score_col = col
                        break
                
                df_list.append({
                    "search_query": search_query,
                    "method": method_name,
                    "rank": rank,
                    "course_code": row['courseCode'],
                    "title": row['title'],
                    "description": row['description'],
                    "score": row[score_col] if score_col else 0
                })
        except Exception as e:
            print(f"Error with method {method_name} on query '{search_query}': {e}")
            continue

# Convert lists to DataFrames **after** data collection
df1 = pd.DataFrame(df_list)


# Export to Excel with 3 sheets
with pd.ExcelWriter(f"recommendations_results_{num}.xlsx") as writer:
    df1.to_excel(writer, sheet_name=f"Query{num}_Results", index=False)




Error with method cosine on query 'Data Visualization': positional indexers are out-of-bounds
Error with method cosine on query 'Data Engineering': positional indexers are out-of-bounds
Error with method bert on query 'Data Engineering': positional indexers are out-of-bounds
